In [25]:
import os
import gc
import glob
import pandas as pd
import numpy as np

In [26]:
# Path containing the 100 merged parquet files
merged_folder = "../data/processed/merged_chunks"

# Get all parquet files
merged_files = sorted(
    glob.glob(os.path.join(merged_folder, "*.parquet"))
)

print("Number of files:", len(merged_files))
print("\nFirst 5 files:")
for file in merged_files[:5]:
    print(os.path.basename(file))

print("\nLast 5 files:")
for file in merged_files[-5:]:
    print(os.path.basename(file))

Number of files: 100

First 5 files:
CA_1_0000.parquet
CA_1_0200.parquet
CA_1_0400.parquet
CA_1_0600.parquet
CA_1_0800.parquet

Last 5 files:
WI_3_1000.parquet
WI_3_1200.parquet
WI_3_1400.parquet
WI_3_1600.parquet
WI_3_1800.parquet


In [27]:
# Load the first merged chunk
sales_long = pd.read_parquet(merged_files[0])
print("File:", os.path.basename(merged_files[0]))
print("Shape:", sales_long.shape)
print("\nColumns:")
print(sales_long.columns.tolist())
print("\nDtypes:")
print(sales_long.dtypes)
print("\nDate range:")
print(sales_long["date"].min(), "→", sales_long["date"].max())

print("\nDay range:")
print(sales_long["d"].min(), "→", sales_long["d"].max())

print("\nMissing values:")
print(sales_long.isna().sum())

File: CA_1_0000.parquet
Shape: (609800, 22)

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price']

Dtypes:
id                object
item_id         category
dept_id         category
cat_id          category
store_id        category
state_id        category
d                  int16
sales              int16
date              object
wm_yr_wk           int16
weekday           object
wday               int64
month              int64
year               int64
event_name_1      object
event_type_1      object
event_name_2      object
event_type_2      object
snap_CA            int64
snap_TX            int64
snap_WI            int64
sell_price       float32
dtype: object

Date range:
2011-01-29 → 2011-08-16

Day range:
1 → 200

Missing values:
id                   0
item_id              0
dept_id  

In [28]:
sales_long

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,1,0,2011-01-29,11101,...,1,2011,None,None,None,None,0,0,0,NaN
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,1,0,2011-01-29,11101,...,1,2011,None,None,None,None,0,0,0,NaN
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,1,0,2011-01-29,11101,...,1,2011,None,None,None,None,0,0,0,NaN
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,1,0,2011-01-29,11101,...,1,2011,None,None,None,None,0,0,0,NaN
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,1,0,2011-01-29,11101,...,1,2011,None,None,None,None,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
609795,FOODS_3_823_CA_1_validation,FOODS_3_823,FOODS_3,FOODS,CA_1,CA,200,0,2011-08-16,11129,...,8,2011,None,None,None,None,0,0,0,NaN
609796,FOODS_3_824_CA_1_validation,FOODS_3_824,FOODS_3,FOODS,CA_1,CA,200,2,2011-08-16,11129,...,8,2011,None,None,None,None,0,0,0,2.48
609797,FOODS_3_825_CA_1_validation,FOODS_3_825,FOODS_3,FOODS,CA_1,CA,200,0,2011-08-16,11129,...,8,2011,None,None,None,None,0,0,0,3.98
609798,FOODS_3_826_CA_1_validation,FOODS_3_826,FOODS_3,FOODS,CA_1,CA,200,0,2011-08-16,11129,...,8,2011,None,None,None,None,0,0,0,NaN


In [29]:
# Convert date to datetime
sales_long["date"] = pd.to_datetime(sales_long["date"])

# Downcast integer columns
sales_long["wday"] = sales_long["wday"].astype("int8")
sales_long["month"] = sales_long["month"].astype("int8")
sales_long["year"] = sales_long["year"].astype("int16")

sales_long["snap_CA"] = sales_long["snap_CA"].astype("int8")
sales_long["snap_TX"] = sales_long["snap_TX"].astype("int8")
sales_long["snap_WI"] = sales_long["snap_WI"].astype("int8")

# Convert repeated text columns to category
category_cols = [
    "id",
    "weekday",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for col in category_cols:
    sales_long[col] = sales_long[col].astype("category")

print(sales_long.dtypes)

id                    category
item_id               category
dept_id               category
cat_id                category
store_id              category
state_id              category
d                        int16
sales                    int16
date            datetime64[ns]
wm_yr_wk                 int16
weekday               category
wday                      int8
month                     int8
year                     int16
event_name_1          category
event_type_1          category
event_name_2          category
event_type_2          category
snap_CA                   int8
snap_TX                   int8
snap_WI                   int8
sell_price             float32
dtype: object


In [30]:
print("Memory usage:")
print(f"{sales_long.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nDate dtype:", sales_long["date"].dtype)

Memory usage:
22.64 MB

Date dtype: datetime64[ns]


In [31]:
# =========================
# TIME FEATURES
# =========================

sales_long["year"] = sales_long["date"].dt.year.astype("int16")

sales_long["month"] = sales_long["date"].dt.month.astype("int8")

sales_long["day_of_week"] = (
    sales_long["date"].dt.dayofweek
).astype("int8")

sales_long["is_weekend"] = (
    sales_long["day_of_week"] >= 5
).astype("int8")

In [32]:
print(
    sales_long[
        ["date", "year", "month", "day_of_week", "is_weekend"]
    ].head(10)
)

        date  year  month  day_of_week  is_weekend
0 2011-01-29  2011      1            5           1
1 2011-01-29  2011      1            5           1
2 2011-01-29  2011      1            5           1
3 2011-01-29  2011      1            5           1
4 2011-01-29  2011      1            5           1
5 2011-01-29  2011      1            5           1
6 2011-01-29  2011      1            5           1
7 2011-01-29  2011      1            5           1
8 2011-01-29  2011      1            5           1
9 2011-01-29  2011      1            5           1


In [33]:
sales_long.shape

(609800, 24)

In [34]:
# =========================
# EVENT FEATURE
# =========================

sales_long["is_event"] = (
    sales_long[["event_name_1", "event_name_2"]]
    .notna()
    .any(axis=1)
    .astype("int8")
)

In [35]:
print(sales_long["is_event"].value_counts())

print(
    sales_long[
        ["date", "event_name_1", "event_name_2", "is_event"]
    ].loc[
        sales_long["is_event"] == 1
    ].head(10)
)

is_event
0    557967
1     51833
Name: count, dtype: int64
            date event_name_1 event_name_2  is_event
24392 2011-02-06    SuperBowl          NaN         1
24393 2011-02-06    SuperBowl          NaN         1
24394 2011-02-06    SuperBowl          NaN         1
24395 2011-02-06    SuperBowl          NaN         1
24396 2011-02-06    SuperBowl          NaN         1
24397 2011-02-06    SuperBowl          NaN         1
24398 2011-02-06    SuperBowl          NaN         1
24399 2011-02-06    SuperBowl          NaN         1
24400 2011-02-06    SuperBowl          NaN         1
24401 2011-02-06    SuperBowl          NaN         1


In [36]:
# =========================
# SNAP FEATURE
# =========================

sales_long["snap_active"] = np.select(
    [
        sales_long["state_id"].eq("CA"),
        sales_long["state_id"].eq("TX"),
        sales_long["state_id"].eq("WI")
    ],
    [
        sales_long["snap_CA"],
        sales_long["snap_TX"],
        sales_long["snap_WI"]
    ],
    default=0
).astype("int8")

In [37]:
print(
    sales_long[
        ["state_id", "snap_CA", "snap_TX", "snap_WI", "snap_active"]
    ].drop_duplicates()
)

      state_id  snap_CA  snap_TX  snap_WI  snap_active
0           CA        0        0        0            0
9147        CA        1        1        0            1
12196       CA        1        0        1            1
15245       CA        1        1        1            1
18294       CA        1        0        0            1
39637       CA        0        1        1            0
45735       CA        0        1        0            0
48784       CA        0        0        1            0


In [38]:
# =========================
# EVENT CATEGORICAL FEATURES
# =========================

event_cols = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for col in event_cols:
    sales_long[col] = (
        sales_long[col]
        .astype("category")
        .cat.add_categories(["No event"])
        .fillna("No event")
    )

In [39]:
print(sales_long[event_cols].dtypes)

print("\nEvent name 1:")
print(sales_long["event_name_1"].value_counts().head(10))

print("\nEvent type 1:")
print(sales_long["event_type_1"].value_counts())

event_name_1    category
event_type_1    category
event_name_2    category
event_type_2    category
dtype: object

Event name 1:
event_name_1
No event          557967
Father's day        3049
ValentinesDay       3049
SuperBowl           3049
StPatricksDay       3049
Ramadan starts      3049
Purim End           3049
PresidentsDay       3049
Pesach End          3049
Cinco De Mayo       3049
Name: count, dtype: int64

Event type 1:
event_type_1
No event     557967
Religious     18294
Cultural      15245
National       9147
Sporting       9147
Name: count, dtype: int64


In [40]:
sales_long["price_available"] = (
    sales_long["sell_price"].notna()
).astype("int8")

In [41]:
print(
    sales_long[
        ["sell_price", "price_available"]
    ].head(20)
)

    sell_price  price_available
0          NaN                0
1          NaN                0
2          NaN                0
3          NaN                0
4          NaN                0
5          NaN                0
6          NaN                0
7         0.46                1
8         1.56                1
9         3.17                1
10         NaN                0
11        5.98                1
12         NaN                0
13         NaN                0
14        0.70                1
15        0.70                1
16         NaN                0
17         NaN                0
18         NaN                0
19         NaN                0


In [42]:
#Define the history requirements
# =========================
# HISTORY CONFIGURATION
# =========================

MAX_LAG = 28
MAX_ROLLING_WINDOW = 28

HISTORY_DAYS = max(MAX_LAG, MAX_ROLLING_WINDOW)

GROUP_COLS = ["item_id", "store_id"]

print("Maximum lag:", MAX_LAG)
print("Maximum rolling window:", MAX_ROLLING_WINDOW)
print("History required:", HISTORY_DAYS, "days")

Maximum lag: 28
Maximum rolling window: 28
History required: 28 days


In [72]:
def prepare_chunk(sales_long, history=None):
    current = sales_long.copy()

    # Mark rows that belong to the current chunk
    current["_is_current"] = 1

    if history is not None and not history.empty:
        history = history.copy()

        # Mark historical rows
        history["_is_current"] = 0

        combined = pd.concat(
            [history, current],
            ignore_index=True
        )
    else:
        combined = current

    combined = combined.sort_values(
        ["item_id", "store_id", "d"]
    ).reset_index(drop=True)

    return combined


def update_history(combined, history_days=28):
    history = (
        combined
        .sort_values(["item_id", "store_id", "d"])
        .groupby(
            ["item_id", "store_id"],
            observed=True,
            group_keys=False
        )
        .tail(history_days)
        .copy()
    )

    # Marker is only needed while creating the current feature chunk
    history.drop(
        columns="_is_current",
        errors="ignore",
        inplace=True
    )

    return history

In [45]:
# =========================
# SALES LAG FEATURES
# =========================

def add_lag_features(combined):
    
    grouped_sales = combined.groupby(
        ["item_id", "store_id"],
        observed=True
    )["sales"]

    combined["sales_lag_1"] = grouped_sales.shift(1)
    combined["sales_lag_7"] = grouped_sales.shift(7)
    combined["sales_lag_28"] = grouped_sales.shift(28)

    return combined

In [46]:
combined = prepare_chunk(sales_long)

combined = add_lag_features(combined)

print(
    combined[
        [
            "item_id",
            "store_id",
            "d",
            "sales",
            "sales_lag_1",
            "sales_lag_7",
            "sales_lag_28"
        ]
    ].head(40)
)

        item_id store_id   d  sales  sales_lag_1  sales_lag_7  sales_lag_28
0   FOODS_1_001     CA_1   1      3          NaN          NaN           NaN
1   FOODS_1_001     CA_1   2      0          3.0          NaN           NaN
2   FOODS_1_001     CA_1   3      0          0.0          NaN           NaN
3   FOODS_1_001     CA_1   4      1          0.0          NaN           NaN
4   FOODS_1_001     CA_1   5      4          1.0          NaN           NaN
5   FOODS_1_001     CA_1   6      2          4.0          NaN           NaN
6   FOODS_1_001     CA_1   7      0          2.0          NaN           NaN
7   FOODS_1_001     CA_1   8      2          0.0          3.0           NaN
8   FOODS_1_001     CA_1   9      0          2.0          0.0           NaN
9   FOODS_1_001     CA_1  10      0          0.0          0.0           NaN
10  FOODS_1_001     CA_1  11      0          0.0          1.0           NaN
11  FOODS_1_001     CA_1  12      0          0.0          4.0           NaN
12  FOODS_1_

In [47]:
# =========================
# ROLLING FEATURES
# =========================

def add_rolling_features(combined):

    grouped_sales = combined.groupby(
        ["item_id", "store_id"],
        observed=True
    )["sales"]

    # Exclude today's sales to prevent data leakage
    previous_sales = grouped_sales.shift(1)

    combined["rolling_mean_7"] = (
        previous_sales
        .groupby(
            [combined["item_id"], combined["store_id"]],
            observed=True
        )
        .transform(
            lambda x: x.rolling(window=7, min_periods=7).mean()
        )
    )

    combined["rolling_mean_28"] = (
        previous_sales
        .groupby(
            [combined["item_id"], combined["store_id"]],
            observed=True
        )
        .transform(
            lambda x: x.rolling(window=28, min_periods=28).mean()
        )
    )

    return combined

In [48]:
combined = add_rolling_features(combined)

print(
    combined[
        [
            "item_id",
            "d",
            "sales",
            "sales_lag_1",
            "rolling_mean_7",
            "rolling_mean_28"
        ]
    ].head(40)
)

        item_id   d  sales  sales_lag_1  rolling_mean_7  rolling_mean_28
0   FOODS_1_001   1      3          NaN             NaN              NaN
1   FOODS_1_001   2      0          3.0             NaN              NaN
2   FOODS_1_001   3      0          0.0             NaN              NaN
3   FOODS_1_001   4      1          0.0             NaN              NaN
4   FOODS_1_001   5      4          1.0             NaN              NaN
5   FOODS_1_001   6      2          4.0             NaN              NaN
6   FOODS_1_001   7      0          2.0             NaN              NaN
7   FOODS_1_001   8      2          0.0        1.428571              NaN
8   FOODS_1_001   9      0          2.0        1.285714              NaN
9   FOODS_1_001  10      0          0.0        1.285714              NaN
10  FOODS_1_001  11      0          0.0        1.285714              NaN
11  FOODS_1_001  12      0          0.0        1.142857              NaN
12  FOODS_1_001  13      3          0.0        0.57

In [49]:
def add_price_features(combined):

    grouped_price = combined.groupby(
        ["item_id", "store_id"],
        observed=True
    )["sell_price"]

    # Previous day's price
    previous_price = grouped_price.shift(1)

    # Absolute price change
    combined["price_change"] = (
        combined["sell_price"] - previous_price
    )

    # Whether the price changed
    combined["price_changed"] = pd.Series(
        pd.NA,
        index=combined.index,
        dtype="Int8"
    )

    valid_price_change = combined["price_change"].notna()

    combined.loc[valid_price_change, "price_changed"] = (
        combined.loc[valid_price_change, "price_change"]
        .ne(0)
        .astype("int8")
    )

    return combined

In [50]:
combined = add_price_features(combined)

print(
    combined[
        [
            "item_id",
            "store_id",
            "d",
            "sell_price",
            "price_change",
            "price_changed"
        ]
    ].head(40)
)

        item_id store_id   d  sell_price  price_change  price_changed
0   FOODS_1_001     CA_1   1         2.0           NaN           <NA>
1   FOODS_1_001     CA_1   2         2.0           0.0              0
2   FOODS_1_001     CA_1   3         2.0           0.0              0
3   FOODS_1_001     CA_1   4         2.0           0.0              0
4   FOODS_1_001     CA_1   5         2.0           0.0              0
5   FOODS_1_001     CA_1   6         2.0           0.0              0
6   FOODS_1_001     CA_1   7         2.0           0.0              0
7   FOODS_1_001     CA_1   8         2.0           0.0              0
8   FOODS_1_001     CA_1   9         2.0           0.0              0
9   FOODS_1_001     CA_1  10         2.0           0.0              0
10  FOODS_1_001     CA_1  11         2.0           0.0              0
11  FOODS_1_001     CA_1  12         2.0           0.0              0
12  FOODS_1_001     CA_1  13         2.0           0.0              0
13  FOODS_1_001     

In [51]:
def add_all_features(combined):

    # -------------------------
    # 1. Lag features
    # -------------------------
    grouped_sales = combined.groupby(
        ["item_id", "store_id"],
        observed=True
    )["sales"]

    combined["sales_lag_1"] = grouped_sales.shift(1)
    combined["sales_lag_7"] = grouped_sales.shift(7)
    combined["sales_lag_28"] = grouped_sales.shift(28)

    # -------------------------
    # 2. Rolling features
    # -------------------------
    combined["rolling_mean_7"] = (
        combined
        .groupby(["item_id", "store_id"], observed=True)["sales"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(7, min_periods=7)
                      .mean()
        )
    )

    combined["rolling_mean_28"] = (
        combined
        .groupby(["item_id", "store_id"], observed=True)["sales"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(28, min_periods=28)
                      .mean()
        )
    )

    # -------------------------
    # 3. Price features
    # -------------------------
    grouped_price = combined.groupby(
        ["item_id", "store_id"],
        observed=True
    )["sell_price"]

    previous_price = grouped_price.shift(1)

    combined["price_change"] = (
        combined["sell_price"] - previous_price
    )

    combined["price_changed"] = pd.Series(
        pd.NA,
        index=combined.index,
        dtype="Int8"
    )

    valid_price_change = combined["price_change"].notna()

    combined.loc[valid_price_change, "price_changed"] = (
        combined.loc[valid_price_change, "price_change"]
        .ne(0)
        .astype("int8")
    )

    return combined

In [52]:
combined = add_all_features(combined)

print(
    combined[
        [
            "item_id",
            "d",
            "sales",
            "sales_lag_1",
            "sales_lag_7",
            "sales_lag_28",
            "rolling_mean_7",
            "rolling_mean_28",
            "sell_price",
            "price_change",
            "price_changed"
        ]
    ].head(40)
)

        item_id   d  sales  sales_lag_1  sales_lag_7  sales_lag_28  \
0   FOODS_1_001   1      3          NaN          NaN           NaN   
1   FOODS_1_001   2      0          3.0          NaN           NaN   
2   FOODS_1_001   3      0          0.0          NaN           NaN   
3   FOODS_1_001   4      1          0.0          NaN           NaN   
4   FOODS_1_001   5      4          1.0          NaN           NaN   
5   FOODS_1_001   6      2          4.0          NaN           NaN   
6   FOODS_1_001   7      0          2.0          NaN           NaN   
7   FOODS_1_001   8      2          0.0          3.0           NaN   
8   FOODS_1_001   9      0          2.0          0.0           NaN   
9   FOODS_1_001  10      0          0.0          0.0           NaN   
10  FOODS_1_001  11      0          0.0          1.0           NaN   
11  FOODS_1_001  12      0          0.0          4.0           NaN   
12  FOODS_1_001  13      3          0.0          2.0           NaN   
13  FOODS_1_001  14 

In [53]:
del combined
gc.collect()

print("Test objects cleared.")

Test objects cleared.


In [54]:
def update_history(combined, history_days=28):
    """
    Keep the latest `history_days` observations
    for each item-store series.
    """

    history = (
        combined
        .sort_values(["item_id", "store_id", "d"])
        .groupby(
            ["item_id", "store_id"],
            observed=True,
            group_keys=False
        )
        .tail(history_days)
        .copy()
    )

    return history

In [55]:
sales_long = pd.read_parquet(merged_files[0])

combined = prepare_chunk(sales_long)

history = update_history(combined)

print("Original rows:", len(combined))
print("History rows:", len(history))

print("\nHistory date range:")
print(history["d"].min(), "→", history["d"].max())

print("\nRows per item-store:")
print(
    history
    .groupby(["item_id", "store_id"], observed=True)
    .size()
    .value_counts()
    .sort_index()
)

Original rows: 609800
History rows: 85372

History date range:
173 → 200

Rows per item-store:
28    3049
Name: count, dtype: int64


In [56]:
# verify cross-chunk lag calculation
sales_long_2 = pd.read_parquet(merged_files[1])

combined_2 = prepare_chunk(
    sales_long_2,
    history=history
)

combined_2 = add_all_features(combined_2)

test_row = combined_2[
    (combined_2["item_id"] == "FOODS_1_001") &
    (combined_2["store_id"] == "CA_1") &
    (combined_2["d"] == 201)
]

print(
    test_row[
        [
            "item_id",
            "store_id",
            "d",
            "sales",
            "sales_lag_1",
            "sales_lag_7",
            "sales_lag_28",
            "rolling_mean_7",
            "rolling_mean_28"
        ]
    ]
)

        item_id store_id    d  sales  sales_lag_1  sales_lag_7  sales_lag_28  \
28  FOODS_1_001     CA_1  201      0          0.0          0.0           1.0   

    rolling_mean_7  rolling_mean_28  
28             0.0         0.392857  


In [57]:
del sales_long_2
del combined_2
del history
gc.collect()

print("Test objects cleared.")

Test objects cleared.


In [58]:
#create feature-engineered output folder
feature_folder = "../data/processed/feature_engineered_chunks"

os.makedirs(feature_folder, exist_ok=True)

print("Output folder:", feature_folder)

Output folder: ../data/processed/feature_engineered_chunks


In [59]:
# function to prepare each chunk

def prepare_sales_chunk(sales_long):
    
    sales_long["date"] = pd.to_datetime(sales_long["date"])

    # Time features
    sales_long["year"] = sales_long["date"].dt.year.astype("int16")
    sales_long["month"] = sales_long["date"].dt.month.astype("int8")
    sales_long["day_of_week"] = (
        sales_long["date"].dt.dayofweek
    ).astype("int8")
    
    sales_long["is_weekend"] = (
        sales_long["day_of_week"] >= 5
    ).astype("int8")

    # Event features
    sales_long["is_event"] = (
        sales_long[["event_name_1", "event_name_2"]]
        .notna()
        .any(axis=1)
        .astype("int8")
    )

    event_cols = [
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2"
    ]

    for col in event_cols:
        sales_long[col] = (
            sales_long[col]
            .astype("category")
            .cat.add_categories(["No event"])
            .fillna("No event")
        )

    # SNAP
    sales_long["snap_active"] = np.select(
        [
            sales_long["state_id"].eq("CA"),
            sales_long["state_id"].eq("TX"),
            sales_long["state_id"].eq("WI")
        ],
        [
            sales_long["snap_CA"],
            sales_long["snap_TX"],
            sales_long["snap_WI"]
        ],
        default=0
    ).astype("int8")

    # Price availability
    sales_long["price_available"] = (
        sales_long["sell_price"].notna()
    ).astype("int8")

    return sales_long

In [60]:
#process the first two chunks
history = None

for i in range(2):

    file = merged_files[i]

    print(f"\nProcessing {os.path.basename(file)}")

    # Load chunk
    sales_long = pd.read_parquet(file)

    # Prepare basic features
    sales_long = prepare_sales_chunk(sales_long)

    # Combine with previous 28-day history
    combined = prepare_chunk(
        sales_long,
        history=history
    )

    # Add lag, rolling and price features
    combined = add_all_features(combined)

    # Keep only the current chunk
    current_d = sales_long["d"].unique()

    feature_chunk = combined[
        combined["d"].isin(current_d)
    ].copy()

    # Save feature-engineered chunk
    output_file = os.path.join(
        feature_folder,
        os.path.basename(file)
    )

    feature_chunk.to_parquet(
        output_file,
        index=False
    )

    # Update history for next chunk
    history = update_history(
        combined,
        history_days=HISTORY_DAYS
    )

    print("Saved:", os.path.basename(output_file))
    print("Feature shape:", feature_chunk.shape)
    print(
        "History:",
        history["d"].min(),
        "→",
        history["d"].max()
    )

    # Cleanup
    del sales_long
    del combined
    del feature_chunk

    gc.collect()


Processing CA_1_0000.parquet
Saved: CA_1_0000.parquet
Feature shape: (609800, 34)
History: 173 → 200

Processing CA_1_0200.parquet
Saved: CA_1_0200.parquet
Feature shape: (609800, 34)
History: 373 → 400


In [61]:
test_feature_file = os.path.join(
    feature_folder,
    "CA_1_0200.parquet"
)

test_features = pd.read_parquet(test_feature_file)

print("Shape:", test_features.shape)

print("\nColumns:")
print(test_features.columns.tolist())

print("\nDtypes:")
print(test_features.dtypes)

print("\nMissing values in feature columns:")
feature_cols = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "price_change",
    "price_changed"
]

print(test_features[feature_cols].isna().sum())

Shape: (609800, 34)

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'day_of_week', 'is_weekend', 'is_event', 'snap_active', 'price_available', 'sales_lag_1', 'sales_lag_7', 'sales_lag_28', 'rolling_mean_7', 'rolling_mean_28', 'price_change', 'price_changed']

Dtypes:
id                         object
item_id                  category
dept_id                  category
cat_id                   category
store_id                 category
state_id                 category
d                           int16
sales                       int16
date               datetime64[ns]
wm_yr_wk                    int16
weekday                    object
wday                        int64
month                        int8
year                        int16
event_name_1               object
event_type_1    

In [76]:
def optimize_feature_dtypes(sales_long):
    category_cols = [
        "id",
        "weekday",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2"
    ]

    for col in category_cols:
        sales_long[col] = sales_long[col].astype("category")

    sales_long["wday"] = sales_long["wday"].astype("int8")
    sales_long["snap_CA"] = sales_long["snap_CA"].astype("int8")
    sales_long["snap_TX"] = sales_long["snap_TX"].astype("int8")
    sales_long["snap_WI"] = sales_long["snap_WI"].astype("int8")

    sales_long["sales_lag_1"] = sales_long["sales_lag_1"].astype("float32")
    sales_long["sales_lag_7"] = sales_long["sales_lag_7"].astype("float32")
    sales_long["sales_lag_28"] = sales_long["sales_lag_28"].astype("float32")
    sales_long["rolling_mean_7"] = sales_long["rolling_mean_7"].astype("float32")
    sales_long["rolling_mean_28"] = sales_long["rolling_mean_28"].astype("float32")
    sales_long["price_change"] = sales_long["price_change"].astype("float32")

    return sales_long

In [63]:
test_features = optimize_feature_dtypes(test_features)

print("Memory usage:")
print(f"{test_features.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nDtypes:")
print(test_features.dtypes)

Memory usage:
73.23 MB

Dtypes:
id                       category
item_id                  category
dept_id                  category
cat_id                   category
store_id                 category
state_id                 category
d                           int16
sales                       int16
date               datetime64[ns]
wm_yr_wk                    int16
weekday                  category
wday                         int8
month                        int8
year                        int16
event_name_1             category
event_type_1             category
event_name_2             category
event_type_2               object
snap_CA                      int8
snap_TX                      int8
snap_WI                      int8
sell_price                float32
day_of_week                  int8
is_weekend                   int8
is_event                     int8
snap_active                  int8
price_available              int8
sales_lag_1               float32
sales_lag_7     

In [64]:
test_features["event_type_2"] = (
    test_features["event_type_2"]
    .astype("category")
)

print("event_type_2 dtype:", test_features["event_type_2"].dtype)

print("\nFinal memory usage:")
print(
    f"{test_features.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

event_type_2 dtype: category

Final memory usage:
40.66 MB


In [65]:
history = None

for i, file in enumerate(merged_files):

    print(
        f"\n[{i + 1}/{len(merged_files)}] "
        f"Processing {os.path.basename(file)}"
    )

    # Load merged chunk
    sales_long = pd.read_parquet(file)

    # Basic feature preparation
    sales_long = prepare_sales_chunk(sales_long)

    # Combine with previous 28 days of history
    combined = prepare _chunk(
        sales_long,
        history=history
    )

    # Add lag, rolling and price features
    combined = add_all_features(combined)

    # Keep only the current chunk
    current_d = sales_long["d"].unique()

    feature_chunk = combined[
        combined["d"].isin(current_d)
    ].copy()

    # Optimize dtypes
    feature_chunk = optimize_feature_dtypes(
        feature_chunk
    )

    # Save
    output_file = os.path.join(
        feature_folder,
        os.path.basename(file)
    )

    feature_chunk.to_parquet(
        output_file,
        index=False
    )

    # Update history for next chunk
    history = update_history(
        combined,
        history_days=HISTORY_DAYS
    )

    print(
        f"Saved: {os.path.basename(output_file)} | "
        f"Shape: {feature_chunk.shape} | "
        f"Memory: "
        f"{feature_chunk.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
    )

    # Cleanup
    del sales_long
    del combined
    del feature_chunk

    gc.collect()

print("\n===================================")
print("ALL CHUNKS PROCESSED SUCCESSFULLY")
print("===================================")


[1/100] Processing CA_1_0000.parquet
Saved: CA_1_0000.parquet | Shape: (609800, 34) | Memory: 40.66 MB

[2/100] Processing CA_1_0200.parquet
Saved: CA_1_0200.parquet | Shape: (609800, 34) | Memory: 77.88 MB

[3/100] Processing CA_1_0400.parquet
Saved: CA_1_0400.parquet | Shape: (609800, 34) | Memory: 77.88 MB

[4/100] Processing CA_1_0600.parquet
Saved: CA_1_0600.parquet | Shape: (609800, 34) | Memory: 77.88 MB

[5/100] Processing CA_1_0800.parquet
Saved: CA_1_0800.parquet | Shape: (609800, 34) | Memory: 77.88 MB

[6/100] Processing CA_1_1000.parquet
Saved: CA_1_1000.parquet | Shape: (609800, 34) | Memory: 77.89 MB

[7/100] Processing CA_1_1200.parquet
Saved: CA_1_1200.parquet | Shape: (609800, 34) | Memory: 77.88 MB

[8/100] Processing CA_1_1400.parquet
Saved: CA_1_1400.parquet | Shape: (609800, 34) | Memory: 77.88 MB

[9/100] Processing CA_1_1600.parquet
Saved: CA_1_1600.parquet | Shape: (609800, 34) | Memory: 110.46 MB

[10/100] Processing CA_1_1800.parquet
Saved: CA_1_1800.parquet

In [66]:
#verify
feature_files = sorted(
    glob.glob(
        os.path.join(feature_folder, "*.parquet")
    )
)

print("Number of feature files:", len(feature_files))

print("\nFirst 5:")
for file in feature_files[:5]:
    print(os.path.basename(file))

print("\nLast 5:")
for file in feature_files[-5:]:
    print(os.path.basename(file))

Number of feature files: 100

First 5:
CA_1_0000.parquet
CA_1_0200.parquet
CA_1_0400.parquet
CA_1_0600.parquet
CA_1_0800.parquet

Last 5:
WI_3_1000.parquet
WI_3_1200.parquet
WI_3_1400.parquet
WI_3_1600.parquet
WI_3_1800.parquet


In [67]:
total_rows = 0
min_d = None
max_d = None

for file in feature_files:
    chunk = pd.read_parquet(
        file,
        columns=["d"]
    )

    total_rows += len(chunk)

    chunk_min = chunk["d"].min()
    chunk_max = chunk["d"].max()

    if min_d is None or chunk_min < min_d:
        min_d = chunk_min

    if max_d is None or chunk_max > max_d:
        max_d = chunk_max

    del chunk

print("Total rows:", total_rows)
print("Minimum d:", min_d)
print("Maximum d:", max_d)

Total rows: 62169110
Minimum d: 1
Maximum d: 1913


In [68]:
duplicate_rows = 0

for file in feature_files:

    chunk = pd.read_parquet(
        file,
        columns=["item_id", "store_id", "d"]
    )

    duplicates = chunk.duplicated(
        subset=["item_id", "store_id", "d"]
    ).sum()

    duplicate_rows += duplicates

    if duplicates > 0:
        print(
            "Duplicates found:",
            os.path.basename(file),
            duplicates
        )

    del chunk

print("\nTotal duplicate rows:", duplicate_rows)


Total duplicate rows: 0


In [69]:
# verify series completeness
series_counts = {}

for file in feature_files:

    chunk = pd.read_parquet(
        file,
        columns=["item_id", "store_id", "d"]
    )

    counts = (
        chunk
        .groupby(
            ["item_id", "store_id"],
            observed=True
        )["d"]
        .nunique()
    )

    for series, count in counts.items():
        series_counts[series] = (
            series_counts.get(series, 0) + count
        )

    del chunk

series_counts = pd.Series(series_counts)

print("Number of item-store series:", len(series_counts))
print("\nDays per series:")
print(series_counts.value_counts().sort_index())

print("\nIncomplete series:", (series_counts != 1913).sum())

Number of item-store series: 30490

Days per series:
1913    3049
1941    3049
1969    3049
1997    3049
2025    3049
2053    3049
2081    3049
2109    3049
2137    3049
2165    3049
Name: count, dtype: int64

Incomplete series: 27441


In [70]:
for file in feature_files[:6]:

    chunk = pd.read_parquet(
        file,
        columns=["d"]
    )

    print(
        os.path.basename(file),
        "rows:", len(chunk),
        "d:", chunk["d"].min(),
        "→",
        chunk["d"].max(),
        "unique d:", chunk["d"].nunique()
    )

    del chunk

CA_1_0000.parquet rows: 609800 d: 1 → 200 unique d: 200
CA_1_0200.parquet rows: 609800 d: 201 → 400 unique d: 200
CA_1_0400.parquet rows: 609800 d: 401 → 600 unique d: 200
CA_1_0600.parquet rows: 609800 d: 601 → 800 unique d: 200
CA_1_0800.parquet rows: 609800 d: 801 → 1000 unique d: 200
CA_1_1000.parquet rows: 609800 d: 1001 → 1200 unique d: 200


In [71]:
for file in feature_files:

    chunk = pd.read_parquet(
        file,
        columns=["d"]
    )

    print(
        f"{os.path.basename(file):25s} "
        f"rows={len(chunk):6d} | "
        f"d={chunk['d'].min():4d} → {chunk['d'].max():4d} | "
        f"unique_d={chunk['d'].nunique():4d}"
    )

    del chunk

CA_1_0000.parquet         rows=609800 | d=   1 →  200 | unique_d= 200
CA_1_0200.parquet         rows=609800 | d= 201 →  400 | unique_d= 200
CA_1_0400.parquet         rows=609800 | d= 401 →  600 | unique_d= 200
CA_1_0600.parquet         rows=609800 | d= 601 →  800 | unique_d= 200
CA_1_0800.parquet         rows=609800 | d= 801 → 1000 | unique_d= 200
CA_1_1000.parquet         rows=609800 | d=1001 → 1200 | unique_d= 200
CA_1_1200.parquet         rows=609800 | d=1201 → 1400 | unique_d= 200
CA_1_1400.parquet         rows=609800 | d=1401 → 1600 | unique_d= 200
CA_1_1600.parquet         rows=609800 | d=1601 → 1800 | unique_d= 200
CA_1_1800.parquet         rows=344537 | d=1801 → 1913 | unique_d= 113
CA_2_0000.parquet         rows=609800 | d=   1 →  200 | unique_d= 200
CA_2_0200.parquet         rows=609800 | d= 201 →  400 | unique_d= 200
CA_2_0400.parquet         rows=609800 | d= 401 →  600 | unique_d= 200
CA_2_0600.parquet         rows=609800 | d= 601 →  800 | unique_d= 200
CA_2_0800.parquet   

In [73]:
test_history = None

for i in range(2):
    file = merged_files[i]

    sales_long = pd.read_parquet(file)
    sales_long = prepare_sales_chunk(sales_long)

    combined = prepare_chunk(
        sales_long,
        history=test_history
    )

    combined = add_all_features(combined)

    feature_chunk = combined[
        combined["_is_current"] == 1
    ].copy()

    print(
        f"Chunk {i+1}:",
        feature_chunk.shape,
        f"d={feature_chunk['d'].min()} → {feature_chunk['d'].max()}",
        f"history_rows={len(test_history) if test_history is not None else 0}"
    )

    test_history = update_history(
        combined,
        history_days=HISTORY_DAYS
    )

    del sales_long, combined, feature_chunk
    gc.collect()

Chunk 1: (609800, 35) d=1 → 200 history_rows=0
Chunk 2: (609800, 35) d=201 → 400 history_rows=85372


In [74]:
# Recalculate the first two chunks for a precise boundary check

test_history = None
test_feature_chunks = []

for i in range(2):
    sales_long = pd.read_parquet(merged_files[i])
    sales_long = prepare_sales_chunk(sales_long)

    combined = prepare_chunk(
        sales_long,
        history=test_history
    )

    combined = add_all_features(combined)

    feature_chunk = combined[
        combined["_is_current"] == 1
    ].copy()

    test_feature_chunks.append(feature_chunk)

    test_history = update_history(
        combined,
        history_days=HISTORY_DAYS
    )

    del sales_long, combined
    gc.collect()

# Pick one item-store series
sample = test_feature_chunks[1][
    (test_feature_chunks[1]["item_id"] == "FOODS_1_001") &
    (test_feature_chunks[1]["store_id"] == "CA_1") &
    (test_feature_chunks[1]["d"] == 201)
]

sample[
    [
        "item_id",
        "store_id",
        "d",
        "sales",
        "sales_lag_1",
        "sales_lag_7",
        "sales_lag_28",
        "rolling_mean_7",
        "rolling_mean_28"
    ]
]

,item_id,store_id,d,sales,sales_lag_1,sales_lag_7,sales_lag_28,rolling_mean_7,rolling_mean_28
28,FOODS_1_001,CA_1,201,0,0.0,0.0,1.0,0.0,0.392857


In [77]:
existing_feature_files = glob.glob(
    os.path.join(feature_folder, "*.parquet")
)

print(f"Found {len(existing_feature_files)} existing feature files.")

for file in existing_feature_files:
    os.remove(file)

print("Old feature files deleted.")

Found 100 existing feature files.
Old feature files deleted.


In [78]:
history = None
current_store = None

for i, file in enumerate(merged_files):

    filename = os.path.basename(file)

    # Example: CA_1_0000.parquet -> CA_1
    store = "_".join(filename.split("_")[:2])

    # Reset history whenever we move to a new store
    if store != current_store:
        history = None
        current_store = store
        print(f"\nStarting store: {store}")

    # Load merged chunk
    sales_long = pd.read_parquet(file)

    # Prepare basic features
    sales_long = prepare_sales_chunk(sales_long)

    # Add previous history from SAME store only
    combined = prepare_chunk(
        sales_long,
        history=history
    )

    # Create lag, rolling and price features
    combined = add_all_features(combined)

    # Keep ONLY rows from the current chunk
    feature_chunk = combined[
        combined["_is_current"] == 1
    ].copy()

    # Remove temporary marker
    feature_chunk.drop(
        columns="_is_current",
        inplace=True
    )

    # Optimize memory
    feature_chunk = optimize_feature_dtypes(
        feature_chunk
    )

    # Save
    output_file = os.path.join(
        feature_folder,
        filename
    )

    feature_chunk.to_parquet(
        output_file,
        index=False
    )

    # Update history for the NEXT chunk
    history = update_history(
        combined,
        history_days=HISTORY_DAYS
    )

    print(
        f"[{i+1}/100] {filename} | "
        f"rows={len(feature_chunk):,} | "
        f"d={feature_chunk['d'].min()}→{feature_chunk['d'].max()}"
    )

    # Free memory
    del sales_long, combined, feature_chunk
    gc.collect()

print("\nFeature engineering completed.")


Starting store: CA_1
[1/100] CA_1_0000.parquet | rows=609,800 | d=1→200
[2/100] CA_1_0200.parquet | rows=609,800 | d=201→400
[3/100] CA_1_0400.parquet | rows=609,800 | d=401→600
[4/100] CA_1_0600.parquet | rows=609,800 | d=601→800
[5/100] CA_1_0800.parquet | rows=609,800 | d=801→1000
[6/100] CA_1_1000.parquet | rows=609,800 | d=1001→1200
[7/100] CA_1_1200.parquet | rows=609,800 | d=1201→1400
[8/100] CA_1_1400.parquet | rows=609,800 | d=1401→1600
[9/100] CA_1_1600.parquet | rows=609,800 | d=1601→1800
[10/100] CA_1_1800.parquet | rows=344,537 | d=1801→1913

Starting store: CA_2
[11/100] CA_2_0000.parquet | rows=609,800 | d=1→200
[12/100] CA_2_0200.parquet | rows=609,800 | d=201→400
[13/100] CA_2_0400.parquet | rows=609,800 | d=401→600
[14/100] CA_2_0600.parquet | rows=609,800 | d=601→800
[15/100] CA_2_0800.parquet | rows=609,800 | d=801→1000
[16/100] CA_2_1000.parquet | rows=609,800 | d=1001→1200
[17/100] CA_2_1200.parquet | rows=609,800 | d=1201→1400
[18/100] CA_2_1400.parquet | rows=6

In [79]:
# Reload feature files
feature_files = sorted(
    glob.glob(os.path.join(feature_folder, "*.parquet"))
)

print("Number of feature files:", len(feature_files))

# --------------------------------------------------
# 1. Total row count
# --------------------------------------------------

total_rows = 0

for file in feature_files:
    total_rows += len(pd.read_parquet(file, columns=["d"]))

print(f"Total rows: {total_rows:,}")
print(f"Expected:   {30_490 * 1_913:,}")


# --------------------------------------------------
# 2. Rows per store
# --------------------------------------------------

store_rows = {}

for file in feature_files:
    temp = pd.read_parquet(
        file,
        columns=["store_id"]
    )

    for store, count in temp["store_id"].value_counts().items():
        store_rows[store] = store_rows.get(store, 0) + count

    del temp

print("\nRows per store:")
for store, count in sorted(store_rows.items()):
    print(f"{store}: {count:,}")


# --------------------------------------------------
# 3. Expected rows per store
# --------------------------------------------------

expected_store_rows = 3_049 * 1_913

print(
    f"\nExpected rows per store: "
    f"{expected_store_rows:,}"
)

print(
    "\nAll stores correct:",
    all(
        count == expected_store_rows
        for count in store_rows.values()
    )
)

Number of feature files: 100
Total rows: 58,327,370
Expected:   58,327,370

Rows per store:
CA_1: 5,832,737
CA_2: 5,832,737
CA_3: 5,832,737
CA_4: 5,832,737
TX_1: 5,832,737
TX_2: 5,832,737
TX_3: 5,832,737
WI_1: 5,832,737
WI_2: 5,832,737
WI_3: 5,832,737

Expected rows per store: 5,832,737

All stores correct: True
